In [ ]:
from codes.utils import Config
import pandas as pd
import re

In [83]:
dir_data = Config.DIR_RESULTS / "amazontextract"
questions_combined = pd.DataFrame()
answers_combined = pd.DataFrame()
for f in dir_data.glob('*.csv'):
    print(f.name)
    if f.name.endswith('_answers.csv'):
        answers = pd.read_csv(f, index_col=0)
        answers['Answer Source'] = f.name.replace('_answers.csv', '.pdf')
        answers_combined = pd.concat([answers_combined, answers])
    elif f.name.endswith('_answers_table.csv'):
        answers = pd.read_csv(f, index_col=0)
        answers['Answers'] = answers[['Guess', 'Actual']].apply(lambda x: x['Actual'] if not pd.isna(x['Actual']) else x['Guess'], axis=1)
        answers['Answer Source'] = f.name.replace('_answers_table.csv', '.pdf')
        answers = answers.drop(columns=['Guess', 'Actual'])
        answers = answers[~pd.isna(answers['Answers'])]
        answers_combined = pd.concat([answers_combined, answers])
    else:
        questions = pd.read_csv(f, index_col=0)
        questions['Question Source'] = f.name.replace('.csv', '.pdf')
        questions_combined = pd.concat([questions_combined, questions])

ABT #17_answers.csv
ABT 2006 recert #23.csv
24-2007-recert no answers.csv
ABT #17.csv
ABT RE No 22 2005.csv
recert 2002- MH group answers_answers_table.csv
ABT2002#19.csv
Document3.csv
ABT #18_answers.csv
Document2.csv
Document6.csv
Document7.csv
ABTexamMay52004 #21.csv
Document5.csv
Document4.csv
Document8.csv
recert18- 2001-MH answers_answers_table.csv
ABT2003#20.csv
ABT2001#18.csv
ABT #19.csv
ABT #18.csv
Document.csv
Document_answers_table.csv
Document7_answers_table.csv
ABT RE No 22 2005 missing page.csv
ABT2000#17.csv
ABT #20.csv


In [84]:
answers_combined[answers_combined['Exam'] == 'Exam booklet page no. 2001'] = 'Exam booklet page no. 2001A'

/var/folders/g3/sdb5v4350551p2bbc02jj5xdwg5ncq/T/ipykernel_86614/911750908.py:1: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'Exam booklet page no. 2001A' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  answers_combined[answers_combined['Exam'] == 'Exam booklet page no. 2001'] = 'Exam booklet page no. 2001A'


In [85]:
def mapVal(x):
    grps = re.findall(r'(Recertification(\ Exam)?\ (\d{4})\ Part\ ([A-Z]).*)|(Exam\ booklet\ page\ no\.\ (\d{4}[A-Z]*))', x)
    if len(grps) == 0 or len(grps[0]) != 6: return x
    if grps[0][0] != '':
        return ''.join(grps[0][2:4])
    return grps[0][5]

In [86]:
questions_combined['Exam'] = questions_combined['Exam'].map(mapVal)
answers_combined['Exam'] = answers_combined['Exam'].map(mapVal)

In [87]:
answers_combined = pd.concat([answers_combined, pd.read_csv(Config.DIR_RESULTS / 'combined_answers_from_nonpdfs.csv', index_col=0)]).reset_index(drop=True)
answers_combined

,Exam,Questions,Answers,Answer Source
0,2000A,1,C,ABT #17.pdf
1,2000A,2,D,ABT #17.pdf
2,2000A,3,B,ABT #17.pdf
3,2000A,4,B,ABT #17.pdf
4,2000A,5,D,ABT #17.pdf
...,...,...,...,...
1368,2007C,36,E,ABT Recert 24-2007 MH group Answers.doc
1369,2007C,37,"A, E",ABT Recert 24-2007 MH group Answers.doc
1370,2007C,38,C,ABT Recert 24-2007 MH group Answers.doc
1371,2007C,39,C,ABT Recert 24-2007 MH group Answers.doc


In [88]:
def extractIndex(x):
    res = re.findall(r'([0-9]{1,2})\..*', x)
    if len(res): return res[0]
    return '?'

In [89]:
questions_combined['Question Index'] = questions_combined[['Exam', 'Questions']].apply(lambda x: f"{x['Exam']} Q{extractIndex(x['Questions'])}", axis=1)
answers_combined['Question Index'] = answers_combined['Exam'] + ' Q' + answers_combined['Questions'].astype(str)

In [90]:
def multipleSelectionCriteria(answer):
    match answer:
        case 'A': return '1, 2, 3'
        case 'B': return '1, 3'
        case 'C': return '2, 4'
        case 'D': return '4'
        case 'E': return '1, 2, 3, 4'

combined_qa = pd.merge(questions_combined[['Question Index', 'Questions', 'Options', 'Question Source']], answers_combined[['Answers', 'Question Index', 'Answer Source']], on='Question Index', how='inner')
combined_qa = combined_qa.drop_duplicates()
combined_qa['Answers'] = combined_qa.apply(lambda x: multipleSelectionCriteria(x['Answers']) if re.match(r'^\d\)', str(x['Options'])) else x['Answers'], axis=1)
def sortQA(x):
    df = x.str.split(' Q', expand=True)
    return df.apply(lambda x: [x[0], int(x[1])], axis=1)
combined_qa = combined_qa.sort_values(by='Question Index', key=sortQA)
combined_qa.reset_index(drop=True).to_csv(Config.DIR_RESULTS / "combined_qa_amazontextract.csv")